## Baselines

In [31]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from src.features import build_features, TARGET_COL
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
import warnings

warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, "..")


def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))


def directional_accuracy(y_true, y_pred):
    """Proporção de acertos na direção da mudança."""
    actual_dir = np.diff(y_true)
    pred_dir = np.diff(y_pred)
    if len(actual_dir) == 0:
        return np.nan
    return np.mean(np.sign(actual_dir) == np.sign(pred_dir))


def diebold_mariano_test(e1, e2, h=1, loss="absolute"):
    """
    Teste de Diebold-Mariano para comparar capacidade preditiva de dois modelos.

    e1, e2: erros de previsão dos modelos 1 e 2
    h: horizonte de previsão (1 para one-step-ahead)
    loss: "absolute" (MAE) ou "squared" (MSE)

    Retorna: estatística DM, p-valor (bilateral)
    H0: os dois modelos têm a mesma capacidade preditiva
    DM > 0 => modelo 2 é melhor (erros menores); DM < 0 => modelo 1 é melhor
    """
    if loss == "absolute":
        d = np.abs(e1) - np.abs(e2)
    else:
        d = e1**2 - e2**2

    n = len(d)
    d_mean = np.mean(d)

    # Autocovariance com Newey-West (até lag h-1)
    gamma_0 = np.var(d, ddof=0)
    gamma_sum = 0
    for k in range(1, h):
        gamma_k = np.mean((d[k:] - d_mean) * (d[:-k] - d_mean))
        gamma_sum += gamma_k

    var_d = (gamma_0 + 2 * gamma_sum) / n

    if var_d <= 0:
        return np.nan, np.nan

    dm_stat = d_mean / np.sqrt(var_d)

    # Correção de Harvey, Leybourne & Newbold (1997) para amostra finita
    correction = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    dm_stat_corrected = dm_stat * correction

    p_value = 2 * stats.t.sf(np.abs(dm_stat_corrected), df=n - 1)

    return dm_stat_corrected, p_value

### Baseline 1 — Persistência (naive)

Nesta célula vamos avaliar o baseline mais simples: prever o valor do mês t como sendo o valor observado no mês anterior. Em outras palavras, a previsão é:

ŷ(t) = y(t−1)

Este baseline não requer treinamento, a previsão é simplesmente o último valor observado, portanto é computado de forma vetorizada sobre todo o período de teste. É o piso mínimo de desempenho: qualquer modelo que não supere a persistência está adicionando mais ruído do que sinal.

In [32]:
# Baseline 1: Persistência (ŷ(t) = y(t-1))

TARGET = "inadimplencia_pf_total"
LAG_COL = "y_lag1"

MIN_TRAIN_MONTHS = 48
df_model = pd.read_parquet("../data/processed/base_modelagem_2.parquet")

test_dates = df_model.index[MIN_TRAIN_MONTHS:]

y_true = df_model.loc[test_dates, TARGET].astype(float).values
y_pred = df_model.loc[test_dates, LAG_COL].astype(float).values  # persistência

metrics_persist = {
    "MAE": mean_absolute_error(y_true, y_pred),
    "RMSE": rmse(y_true, y_pred),
    "DA": directional_accuracy(y_true, y_pred),
    "N": len(y_true),
    "Período início": test_dates.min().date(),
    "Período fim": test_dates.max().date(),
}

metrics_persist

{'MAE': 0.08142857142857143,
 'RMSE': np.float64(0.10600539069850579),
 'DA': np.float64(0.5865384615384616),
 'N': 105,
 'Período início': datetime.date(2017, 4, 1),
 'Período fim': datetime.date(2025, 12, 1)}

#### DA (Directional Accuracy)

A DA mede a proporção de vezes que o modelo acertou a direção da mudança (subiu ou desceu) da inadimplência de um mês para o outro. O cálculo é:

$$DA = \frac{1}{N-1} \sum_{t=2}^{N} \mathbb{1}\left[\text{sign}(\hat{y}t - \hat{y}{t-1}) = \text{sign}(y_t - y_{t-1})\right]$$

Em palavras simples: se a inadimplência subiu de um mês para o outro, o modelo previu que subiria? Se desceu, previu que desceria?

### Baseline 2 — AR(1) linear (regressão com y(t−1))

O baseline de persistência assume que o valor futuro será idêntico ao atual, ou seja, ŷ(t) = y(t−1). Essa é uma hipótese forte: implica coeficiente angular igual a 1 e intercepto igual a 0. O AR(1) linear relaxa essa restrição ao estimar ambos os parâmetros a partir dos dados:

$$y(t) = a + b \cdot y(t-1)$$

Na prática, o modelo aprende duas coisas que a persistência ignora: um nível médio de longo prazo (capturado por *a*) e a intensidade com que desvios recentes se propagam para o próximo período (capturada por *b*). Se os dados confirmarem a hipótese de persistência, o ajuste convergirá naturalmente para a ≈ 0 e b ≈ 1, portanto o AR(1) nunca será estruturalmente pior, e qualquer ganho de desempenho representará evidência de que a dinâmica da série vai além da simples repetição do último valor.

A avaliação segue o mesmo esquema walk-forward com janela expansível: para cada mês *t*, o modelo é treinado com todos os dados até *t−1* e produz uma única previsão para *t*. Isso garante comparabilidade direta com o baseline de persistência, mesmas datas de teste, mesmas métricas, sem vazamento de informação futura.

In [33]:
# Baseline 2 AR(1) linear

X_COL = [LAG_COL]

y_true_list = []
y_pred_list = []
dates = []

for t in test_dates:
    train = df_model.loc[:t].iloc[:-1]  # até t-1
    test = df_model.loc[[t]]            # t

    lr = LinearRegression()
    lr.fit(train[X_COL], train[TARGET])

    pred = float(lr.predict(test[X_COL])[0])

    y_true_list.append(float(test[TARGET].iloc[0]))
    y_pred_list.append(pred)
    dates.append(t)

results_ar1 = pd.DataFrame(
    {"y_true": y_true_list, "y_pred": y_pred_list},
    index=pd.to_datetime(dates),
)

metrics_ar1 = {
    "MAE": mean_absolute_error(results_ar1["y_true"], results_ar1["y_pred"]),
    "RMSE": rmse(results_ar1["y_true"], results_ar1["y_pred"]),
    "DA": directional_accuracy(results_ar1["y_true"].values, results_ar1["y_pred"].values),
    "N": len(results_ar1),
    "Período início": results_ar1.index.min().date(),
    "Período fim": results_ar1.index.max().date(),
}

metrics_ar1

{'MAE': 0.0855362045204738,
 'RMSE': np.float64(0.11113887528607667),
 'DA': np.float64(0.5961538461538461),
 'N': 105,
 'Período início': datetime.date(2017, 4, 1),
 'Período fim': datetime.date(2025, 12, 1)}

### Baseline 3 — ARIMA(1,1,0) univariado

Terceiro baseline da avaliação, o ARIMA(1,1,0) é um modelo estatístico clássico para séries temporais que utiliza exclusivamente o histórico da variável-alvo. Seu objetivo é capturar a dinâmica temporal da série, incluindo possíveis não estacionaridades, tratadas por meio de diferenciação, e gerar previsões um passo à frente. A avaliação segue o mesmo esquema *walk-forward* com janela expansível adotado nos baselines anteriores: para cada mês $t$, o modelo é ajustado com os dados disponíveis até $t-1$ e produz a previsão $\hat{y}(t)$. Os erros são mensurados por MAE e RMSE, permitindo comparação direta com os baselines precedentes.

O modelo ARIMA é definido por três componentes, representados pelos parâmetros $(p, d, q)$:

- **AR (Autorregressivo — $p$):** modela a dependência linear entre a observação corrente e $p$ observações passadas (lags), capturando a persistência temporal da série.
- **I (Integrado — $d$):** aplica $d$ diferenciações aos dados brutos para induzir estacionaridade, removendo tendências sistemáticas de nível.
- **MA (Média Móvel — $q$):** modela a dependência entre a observação corrente e $q$ erros de previsão passados (resíduos), capturando choques transitórios não explicados pelo componente autorregressivo.

In [34]:
# Baseline 3: ARIMA(1,1,0) univariado com walk-forward expanding window

y_true_list = []
y_pred_list = []
dates = []

# Série alvo completa no df_model
y_series = df_model[TARGET].astype(float)

for t in test_dates:
    train_y = y_series.loc[:t].iloc[:-1]  # até t-1
    test_y = y_series.loc[t]              # y(t)

    # ARIMA(1,1,0) treinado só no histórico do alvo
    model = ARIMA(train_y, order=(1, 1, 0))
    fit = model.fit()

    pred = float(fit.forecast(steps=1).iloc[0])

    y_true_list.append(float(test_y))
    y_pred_list.append(pred)
    dates.append(t)

results_arima110 = pd.DataFrame(
    {"y_true": y_true_list, "y_pred": y_pred_list},
    index=pd.to_datetime(dates),
)

metrics_arima110 = {
    "MAE": mean_absolute_error(results_arima110["y_true"], results_arima110["y_pred"]),
    "RMSE": rmse(results_arima110["y_true"], results_arima110["y_pred"]),
    "DA": directional_accuracy(results_arima110["y_true"].values, results_arima110["y_pred"].values),
    "N": len(results_arima110),
    "Período início": results_arima110.index.min().date(),
    "Período fim": results_arima110.index.max().date(),
}

metrics_arima110

{'MAE': 0.07793545900213478,
 'RMSE': np.float64(0.1031298829754279),
 'DA': np.float64(0.6153846153846154),
 'N': 105,
 'Período início': datetime.date(2017, 4, 1),
 'Período fim': datetime.date(2025, 12, 1)}

Com base no backtesting walk-forward (janela expansível) no período de 2017-04 a 2025-12 (105 previsões), observamos que:

- O baseline de persistência (prever y(t) = y(t-1)) já apresenta bom desempenho, com MAE de 0,0814 e RMSE de 0,1060, indicando forte inércia no target.

- O baseline AR(1) linear (regressão usando apenas y(t-1)) teve desempenho inferior à persistência (MAE 0,0855; RMSE 0,1111), sugerindo que o ajuste mensal do coeficiente adiciona ruído e não melhora a previsão em relação ao naive.

- O ARIMA(1,1,0) univariado foi o melhor baseline entre os testados, superando a persistência (MAE 0,0779; RMSE 0,1031). Assim, ele se torna o principal benchmark a ser batido pelos modelos com variáveis exógenas.

- A **Directional Accuracy** (acerto de direção) dos baselines permite avaliar se cada modelo acerta ao menos a tendência de subida/descida da inadimplência, métrica mais relevante que o erro absoluto para decisores de política econômica e gestores de risco.

| Modelo | MAE | RMSE | DA |
|---|---|---|---|
| ARIMA(1,1,0) | 0.0779 | 0.1031 | 61.5% |
| Persistência | 0.0814 | 0.1060 | 58.7% |
| AR(1) | 0.0855 | 0.1111 | 59.6% |


A partir daqui, a hipótese central passa a ser: variáveis macroeconômicas e financeiras só serão consideradas úteis se conseguirem melhorar de forma consistente o desempenho fora da amostra em relação ao ARIMA(1,1,0) univariado. Para validar essa referência, aplicamos a seguir o teste de Diebold-Mariano entre os próprios baselines.

### Teste de Diebold-Mariano entre Baselines

Até aqui comparamos os baselines por MAE, RMSE e DA. Mas será que a diferença entre o ARIMA(1,1,0) e a Persistência (MAE 0,0779 vs 0,0814 — diferença de apenas 0,0035) é **estatisticamente significativa**, ou pode ser explicada pelo acaso?

O **teste de Diebold-Mariano (DM)** (Diebold & Mariano, 1995) responde exatamente essa pergunta. A ideia é simples:

1. Para cada mês $t$, calculamos o erro absoluto de cada modelo: $|e_1(t)|$ e $|e_2(t)|$
2. Calculamos a diferença: $d(t) = |e_1(t)| - |e_2(t)|$
3. Se $d(t)$ é sistematicamente positivo, o modelo 2 erra menos; se negativo, o modelo 1 erra menos
4. O teste verifica se a média de $d(t)$ é significativamente diferente de zero

**Hipóteses:**
- $H_0$: Os dois modelos têm a mesma capacidade preditiva (média de $d = 0$)
- $H_1$: Os modelos diferem em capacidade preditiva (média de $d \neq 0$)

**Estatística do teste:**

$$DM = \frac{\bar{d}}{\sqrt{\hat{V}(\bar{d})}}$$

onde $\bar{d}$ é a média das diferenças de perda e $\hat{V}(\bar{d})$ é a variância estimada via Newey-West (que corrige para autocorrelação nos erros de previsão). Sob $H_0$, a estatística segue uma distribuição $t$ de Student com $n-1$ graus de liberdade.

**Correção de Harvey, Leybourne & Newbold (1997):** Para amostras finitas (como nossas 105 observações), a estatística DM original tende a rejeitar $H_0$ com mais frequência do que deveria (tamanho efetivo do teste maior que o nominal). A correção HLN ajusta a estatística multiplicando por um fator $\sqrt{(n + 1 - 2h + h(h-1)/n) / n}$ que depende do tamanho da amostra $n$ e do horizonte de previsão $h$, tornando o teste mais conservador e confiável.

**Interpretação:** Se $p < 0.05$, rejeitamos $H_0$ — a diferença de desempenho é estatisticamente significativa, não é artefato amostral.

**Por que incluir este teste já nos baselines?** Porque o baseline ARIMA(1,1,0) será o benchmark de referência para todo o restante do trabalho. Se a diferença entre os baselines não fosse significativa, qualquer modelo exógeno que superasse o ARIMA por margem similar também poderia ser um artefato amostral. Confirmar a significância aqui valida a sensibilidade da metodologia de avaliação e estabelece o limiar de significância prática.

In [35]:
# Teste de Diebold-Mariano entre baselines

# Erros de previsão de cada baseline
e_persist = y_true - y_pred  # persistência (arrays do cell 3)
e_ar1 = results_ar1["y_true"].values - results_ar1["y_pred"].values
e_arima = results_arima110["y_true"].values - results_arima110["y_pred"].values

# Comparações par-a-par
comparisons = [
    ("Persistência", "ARIMA(1,1,0)", e_persist, e_arima),
    ("AR(1)", "ARIMA(1,1,0)", e_ar1, e_arima),
    ("Persistência", "AR(1)", e_persist, e_ar1),
]

dm_baseline_rows = []
for name1, name2, errors1, errors2 in comparisons:
    dm_stat, p_val = diebold_mariano_test(errors1, errors2, h=1)
    # DM > 0 => modelo 2 (segundo argumento) tem erros menores
    if not np.isnan(dm_stat):
        melhor = name2 if dm_stat > 0 else name1
    else:
        melhor = "—"

    dm_baseline_rows.append({
        "Modelo 1": name1,
        "Modelo 2": name2,
        "DM stat": round(dm_stat, 4) if not np.isnan(dm_stat) else np.nan,
        "p-valor": round(p_val, 4) if not np.isnan(p_val) else np.nan,
        "Significativo (5%)": "Sim" if not np.isnan(p_val) and p_val < 0.05 else "Não",
        "Melhor": melhor,
    })

dm_baselines = pd.DataFrame(dm_baseline_rows)
print("Teste de Diebold-Mariano (bilateral, loss=MAE, correção HLN)")
print("DM > 0 => Modelo 2 é melhor | DM < 0 => Modelo 1 é melhor\n")
dm_baselines

Teste de Diebold-Mariano (bilateral, loss=MAE, correção HLN)
DM > 0 => Modelo 2 é melhor | DM < 0 => Modelo 1 é melhor



,Modelo 1,Modelo 2,DM stat,p-valor,Significativo (5%),Melhor
0,Persistência,"ARIMA(1,1,0)",2.6044,0.0105,Sim,"ARIMA(1,1,0)"
1,AR(1),"ARIMA(1,1,0)",2.8621,0.0051,Sim,"ARIMA(1,1,0)"
2,Persistência,AR(1),-2.0881,0.0392,Sim,Persistência


1. ARIMA(1,1,0) vs Persistência (DM=2.60, p=0.011): ARIMA é significativamente melhor. A diferença de MAE (0.0035) não é acaso.
2. ARIMA(1,1,0) vs AR(1) (DM=2.86, p=0.005): ARIMA é significativamente melhor. Resultado ainda mais forte.
3. Persistência vs AR(1) (DM=-2.09, p=0.039): Persistência é significativamente melhor que AR(1). Confirma que estimar coeficientes via OLS com janela expansível
adiciona ruído — o ajuste mensal de α e β é instável e prejudica a previsão.

Isso valida a hierarquia: ARIMA(1,1,0) > Persistência > AR(1), e as três diferenças são estatisticamente significativas a 5%. Isso confirma que ARIMA(1, 1, 0) é o benchmark a ser batido para o restante da análise.

In [36]:
dm_baselines.to_clipboard()

## Modelos com Variáveis Exógenas

### SARIMAX/ARIMAX — ARIMA(1,1,0) com variáveis exógenas

Vamos avaliar um modelo com variáveis exógenas (ARIMAX/SARIMAX) usando o mesmo protocolo walk-forward (janela expansível). A ideia é verificar se as features macro/financeiras (já defasadas no df_model, portanto sem vazamento) trazem ganho preditivo fora da amostra em relação ao baseline ARIMA(1,1,0) univariado. Manteremos a ordem (1,1,0) para comparabilidade direta.

In [37]:
# SARIMAX: ARIMA(1,1,0) + exógenas com walk-forward expanding window

# X exógenas: todas as colunas exceto alvo
X_cols = [c for c in df_model.columns if c != TARGET]

y_true_list = []
y_pred_list = []
dates = []

y_series = df_model[TARGET].astype(float)

for t in test_dates:
    train = df_model.loc[:t].iloc[:-1]   # até t-1
    test  = df_model.loc[[t]]            # t

    train_y = train[TARGET].astype(float)
    train_X = train[X_cols].astype(float)

    test_X = test[X_cols].astype(float)

    # SARIMAX com order (1,1,0)
    model = SARIMAX(
        train_y,
        exog=train_X,
        order=(1, 1, 0),
        trend="c",                 # intercepto (pode ajustar depois)
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fit = model.fit(disp=False)

    pred = float(fit.forecast(steps=1, exog=test_X).iloc[0])

    y_true_list.append(float(test[TARGET].iloc[0]))
    y_pred_list.append(pred)
    dates.append(t)

results_sarimax = pd.DataFrame(
    {"y_true": y_true_list, "y_pred": y_pred_list},
    index=pd.to_datetime(dates),
)

metrics_sarimax = {
    "MAE": mean_absolute_error(results_sarimax["y_true"], results_sarimax["y_pred"]),
    "RMSE": rmse(results_sarimax["y_true"], results_sarimax["y_pred"]),
    "DA": directional_accuracy(results_sarimax["y_true"].values, results_sarimax["y_pred"].values),
    "N": len(results_sarimax),
    "Período início": results_sarimax.index.min().date(),
    "Período fim": results_sarimax.index.max().date(),
}

metrics_sarimax

{'MAE': 0.08010503038935922,
 'RMSE': np.float64(0.10575498044242322),
 'DA': np.float64(0.5961538461538461),
 'N': 105,
 'Período início': datetime.date(2017, 4, 1),
 'Período fim': datetime.date(2025, 12, 1)}

Mini-tuning do SARIMAX (walk-forward)

Nesta etapa vamos testar pequenas variações do SARIMAX para verificar se a piora observada com exógenas foi causada por especificação inadequada. Vamos manter o mesmo protocolo walk-forward (janela expansível) e comparar combinações simples de: (i) ordem ARIMA (p,d,q), (ii) uso ou não de intercepto (trend) e (iii) subconjuntos parcimoniosos de variáveis exógenas. O objetivo não é fazer um grid grande, mas sim checar ajustes de baixo custo que podem estabilizar o modelo e, se possível, superar o ARIMA(1,1,0) univariado.

In [38]:
# Mini-tuning SARIMAX com walk-forward expanding window

# 1) Definir conjuntos de exógenas (parsimoniosos -> completo)
EXOG_SETS = {
    "selic+juros": ["selic_lag8", "juros_pf_lag6"],
    "selic+juros+ibc": ["selic_lag8", "juros_pf_lag6", "ibc_lag12"],
    "all_exog": [c for c in df_model.columns if c not in [TARGET]],
}

# 2) Pequeno grid de ordens e trend
ORDERS = [(1,1,0), (0,1,1), (1,1,1)]
TRENDS = ["n", "c"]  # sem intercepto vs com intercepto

def walkforward_sarimax(order, trend, exog_cols, min_train_months=MIN_TRAIN_MONTHS):
    test_dates_local = df_model.index[min_train_months:]
    y_true, y_pred = [], []

    for t in test_dates_local:
        train = df_model.loc[:t].iloc[:-1]
        test  = df_model.loc[[t]]

        train_y = train[TARGET].astype(float)
        train_X = train[exog_cols].astype(float)
        test_X  = test[exog_cols].astype(float)

        model = SARIMAX(
            train_y,
            exog=train_X,
            order=order,
            trend=trend,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fit = model.fit(disp=False)

        pred = float(fit.forecast(steps=1, exog=test_X).iloc[0])
        y_true.append(float(test[TARGET].iloc[0]))
        y_pred.append(pred)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return {
        "order": str(order),
        "trend": trend,
        "exog_set": None,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "DA": directional_accuracy(y_true, y_pred),
        "N": len(y_true),
    }

# 3) Rodar grid
rows = []
for exog_name, exog_cols in EXOG_SETS.items():
    for order in ORDERS:
        for trend in TRENDS:
            try:
                out = walkforward_sarimax(order=order, trend=trend, exog_cols=exog_cols)
                out["exog_set"] = exog_name
                rows.append(out)
            except Exception as e:
                rows.append({
                    "order": str(order),
                    "trend": trend,
                    "exog_set": exog_name,
                    "MAE": np.nan,
                    "RMSE": np.nan,
                    "DA": np.nan,
                    "N": 0,
                    "error": str(e)[:200],
                })

results_grid = pd.DataFrame(rows).sort_values(["MAE", "RMSE"])
results_grid

,order,trend,exog_set,MAE,RMSE,DA,N
4,"(1, 1, 1)",n,selic+juros,0.074923,0.100781,0.596154,105
5,"(1, 1, 1)",c,selic+juros,0.075565,0.101380,0.596154,105
10,"(1, 1, 1)",n,selic+juros+ibc,0.076105,0.102035,0.596154,105
2,"(0, 1, 1)",n,selic+juros,0.076357,0.102449,0.576923,105
0,"(1, 1, 0)",n,selic+juros,0.076475,0.103069,0.586538,105
11,"(1, 1, 1)",c,selic+juros+ibc,0.077095,0.102189,0.596154,105
6,"(1, 1, 0)",n,selic+juros+ibc,0.077431,0.103870,0.586538,105
8,"(0, 1, 1)",n,selic+juros+ibc,0.077615,0.103451,0.576923,105
14,"(0, 1, 1)",n,all_exog,0.078514,0.102376,0.605769,105
3,"(0, 1, 1)",c,selic+juros,0.078604,0.103566,0.586538,105


## SARIMAX com componente sazonal

Inadimplência pode ter sazonalidade (13º salário em dezembro reduz inadimplência, início de ano aumenta). Vamos testar o melhor modelo SARIMAX encontrado acima, ARIMA(1,1,1) com `selic_lag8 + juros_pf_lag6`, adicionando seasonal_order para verificar se há ganho sazonal.

In [39]:
# SARIMAX com seasonal_order — walk-forward expanding window

SEASONAL_ORDERS = [
    (1, 0, 0, 12),
    (0, 1, 1, 12),
    (1, 1, 0, 12),
]
BEST_EXOG = ["selic_lag8", "juros_pf_lag6"]
BEST_ORDER = (1, 1, 1)

rows_seasonal = []
for sorder in SEASONAL_ORDERS:
    y_true_s, y_pred_s = [], []
    for t in test_dates:
        train = df_model.loc[:t].iloc[:-1]
        test_row = df_model.loc[[t]]
        train_y = train[TARGET].astype(float)
        train_X = train[BEST_EXOG].astype(float)
        test_X = test_row[BEST_EXOG].astype(float)

        try:
            model = SARIMAX(
                train_y, exog=train_X,
                order=BEST_ORDER, seasonal_order=sorder,
                trend="n",
                enforce_stationarity=False, enforce_invertibility=False,
            )
            fit = model.fit(disp=False, maxiter=200)
            pred = float(fit.forecast(steps=1, exog=test_X).iloc[0])
        except Exception:
            pred = np.nan

        y_true_s.append(float(test_row[TARGET].iloc[0]))
        y_pred_s.append(pred)

    y_true_s = np.array(y_true_s)
    y_pred_s = np.array(y_pred_s)
    valid = ~np.isnan(y_pred_s)

    rows_seasonal.append({
        "seasonal_order": str(sorder),
        "MAE": mean_absolute_error(y_true_s[valid], y_pred_s[valid]),
        "RMSE": rmse(y_true_s[valid], y_pred_s[valid]),
        "DA": directional_accuracy(y_true_s[valid], y_pred_s[valid]),
        "N_valid": int(valid.sum()),
    })

results_seasonal = pd.DataFrame(rows_seasonal).sort_values("MAE")
print("Melhor SARIMAX não-sazonal (referência): MAE = 0.0749")
results_seasonal

Melhor SARIMAX não-sazonal (referência): MAE = 0.0749


,seasonal_order,MAE,RMSE,DA,N_valid
0,"(1, 0, 0, 12)",0.071419,0.095205,0.586538,105
2,"(1, 1, 0, 12)",0.076122,0.092587,0.644231,105
1,"(0, 1, 1, 12)",23367.394398,237098.912826,0.644231,105


### Entendendo os parâmetros do SARIMAX

O SARIMAX tem **dois conjuntos independentes** de parâmetros que coexistem:

```
SARIMAX(order=(1,1,1), seasonal_order=(0,1,1,12))
        ├─────────┘    ├──────────────────────┘
        Parte NÃO      Parte SAZONAL
        sazonal
```

**`order = (p, d, q)`** — componente regular, opera sobre meses consecutivos (t-1, t-2, ...):

| Param | Valor | O que faz |
|---|---|---|
| p=1 | AR(1) | Usa $y_{t-1}$ para prever |
| d=1 | Diferenciação | Transforma $y_t$ em $y_t - y_{t-1}$ |
| q=1 | MA(1) | Usa o erro $\varepsilon_{t-1}$ para corrigir |

**`seasonal_order = (P, D, Q, s)`** — componente sazonal, opera sobre múltiplos do período sazonal (t-12, t-24, ...):

| Param | Valor | O que faz |
|---|---|---|
| P | AR sazonal | Usa $y_{t-12}$ como preditor |
| D | Diferenciação sazonal | Transforma $y_t$ em $y_t - y_{t-12}$ |
| Q | MA sazonal | Usa o erro $\varepsilon_{t-12}$ para corrigir |
| s=12 | Período | Define que "sazonal" = 12 meses |

Os dois conjuntos **coexistem e se multiplicam**. O `order=(1,1,1)` vem do mini-tuning realizado acima (melhor configuração com MAE=0.0749), e na célula acima testamos diferentes `seasonal_order` em cima desse `BEST_ORDER`. O modelo completo aplica **ambos** simultaneamente:

```
Diferenciação regular:  Δyₜ = yₜ - yₜ₋₁           (d=1, do order)
Diferenciação sazonal:  Δ₁₂yₜ = yₜ - yₜ₋₁₂        (D=1, do seasonal_order)
Combinação:  ΔΔ₁₂yₜ = (yₜ - yₜ₋₁) - (yₜ₋₁₂ - yₜ₋₁₃)   ← duas diferenciações!
```

### Por que o modelo (0,1,1,12) divergiu?

| Modelo | order | seasonal_order | Total de diferenciações | Resultado |
|---|---|---|---|---|
| A | (1,1,1) | **(1,0,0,12)** | d=1 apenas | MAE=0.071 |
| B | (1,1,1) | **(1,1,0,12)** | d=1 + D=1 = **2** | MAE=0.076 |
| C | (1,1,1) | **(0,1,1,12)** | d=1 + D=1 = **2** | MAE=23.367 |

O problema está no **D=1** (diferenciação sazonal). Como o modelo já tem **d=1** no `order`, adicionar **D=1** no `seasonal_order` resulta em **sobre-diferenciação** — estamos "limpando" a série mais do que o necessário. O efeito é como ajustar o foco de uma câmera além do ponto ideal: em vez de ficar mais nítido, fica borrado novamente.

A decomposição STL na EDA mostrou que a sazonalidade da inadimplência PF responde por apenas **~1% da variância** total. Quando aplicamos D=1, estamos dizendo ao modelo "remova o padrão sazonal por diferenciação" — mas quase não há padrão sazonal para remover. O que acontece na prática:

1. A diferenciação sazonal **remove sinal útil** (tendência e autocorrelação) em vez de sazonalidade
2. A série sobre-diferenciada vira basicamente **ruído amplificado**
3. O componente MA sazonal (Q=1) tenta modelar esse ruído, mas os coeficientes se tornam **numericamente instáveis**
4. Na hora de prever, esses coeficientes instáveis **explodem** → MAE = 23.367 (vs ~0.07 dos modelos estáveis)

Os modelos B e C ambos sobre-diferenciam (d+D=2), mas C diverge porque além de sobre-diferenciar, tenta modelar o ruído resultante com MA sazonal (Q=1), cujos coeficientes se tornam instáveis.

### Por que (1,0,0,12) funciona bem?

| | (0,1,1,12) — divergiu | (1,0,0,12) — melhor modelo |
|---|---|---|
| D (diferenciação sazonal) | **1** — sobre-diferencia | **0** — não diferencia |
| Como captura sazonalidade | Subtrai $y_{t-12}$ da série | Usa $y_{t-12}$ como **preditor** (AR sazonal) |
| Analogia | "Apaga" a sazonalidade antes de modelar | "Aprende" da sazonalidade sem destruí-la |
| Resultado | MAE = 23.367 | MAE = 0.071 |

O modelo `(1,0,0,12)` usa P=1 (um lag AR sazonal), que adiciona $y_{t-12}$ como variável explicativa. Isso captura qualquer padrão sazonal residual de forma **aditiva e suave**, sem a agressividade de diferenciar a série inteira. É a abordagem correta quando a sazonalidade existe mas é fraca.

**Regra prática**: D=1 (diferenciação sazonal) só se justifica quando a série tem sazonalidade **forte e dominante** (ex: vendas de varejo com picos natalinos claros). Para séries onde a tendência domina e a sazonalidade é secundária — como a inadimplência PF — usar P≥1 com D=0 é mais seguro e eficaz.

### Teste de Diebold-Mariano: modelos com exógenas

Agora que temos os três modelos SARIMAX avaliados [ARIMA(1,1,0)] univariado (baseline), SARIMAX(1,1,1) com exógenas sem sazonalidade (mini-tuning), e SARIMAX(1,1,1)(1,0,0)₁₂ com exógenas e componente sazonal. Aplicamos o teste de Diebold-Mariano para verificar se as diferenças de desempenho são estatisticamente significativas.

In [40]:
# Teste de Diebold-Mariano: ARIMA vs SARIMAX (com e sem sazonalidade)

# Re-executar SARIMAX(1,1,1) SEM sazonalidade para ter previsões comparáveis
sarimax_noseas_true, sarimax_noseas_pred = [], []
for t in test_dates:
    train = df_model.loc[:t].iloc[:-1]
    test_row = df_model.loc[[t]]
    train_y = train[TARGET].astype(float)
    train_X = train[BEST_EXOG].astype(float)
    test_X = test_row[BEST_EXOG].astype(float)

    model = SARIMAX(
        train_y, exog=train_X,
        order=BEST_ORDER, trend="n",
        enforce_stationarity=False, enforce_invertibility=False,
    )
    fit = model.fit(disp=False)
    pred = float(fit.forecast(steps=1, exog=test_X).iloc[0])

    sarimax_noseas_true.append(float(test_row[TARGET].iloc[0]))
    sarimax_noseas_pred.append(pred)

sarimax_noseas_true = np.array(sarimax_noseas_true)
sarimax_noseas_pred = np.array(sarimax_noseas_pred)

# Re-executar SARIMAX(1,1,1)(1,0,0)₁₂ COM sazonalidade
sarimax_seas_true, sarimax_seas_pred = [], []
for t in test_dates:
    train = df_model.loc[:t].iloc[:-1]
    test_row = df_model.loc[[t]]
    train_y = train[TARGET].astype(float)
    train_X = train[BEST_EXOG].astype(float)
    test_X = test_row[BEST_EXOG].astype(float)

    try:
        model = SARIMAX(
            train_y, exog=train_X,
            order=BEST_ORDER, seasonal_order=(1, 0, 0, 12),
            trend="n",
            enforce_stationarity=False, enforce_invertibility=False,
        )
        fit = model.fit(disp=False, maxiter=200)
        pred = float(fit.forecast(steps=1, exog=test_X).iloc[0])
    except Exception:
        pred = np.nan

    sarimax_seas_true.append(float(test_row[TARGET].iloc[0]))
    sarimax_seas_pred.append(pred)

sarimax_seas_true = np.array(sarimax_seas_true)
sarimax_seas_pred = np.array(sarimax_seas_pred)

# Erros de previsão
e_arima = results_arima110["y_true"].values - results_arima110["y_pred"].values
e_sarimax_noseas = sarimax_noseas_true - sarimax_noseas_pred

# Para o sazonal, filtrar NaN
valid_seas = ~np.isnan(sarimax_seas_pred)
e_sarimax_seas = sarimax_seas_true[valid_seas] - sarimax_seas_pred[valid_seas]
e_arima_seas = e_arima[valid_seas]
e_noseas_seas = e_sarimax_noseas[valid_seas]

# Comparações par-a-par
comparisons = [
    ("ARIMA(1,1,0)", "SARIMAX(1,1,1)", e_arima, e_sarimax_noseas),
    ("ARIMA(1,1,0)", "SARIMAX(1,1,1)(1,0,0)₁₂", e_arima_seas, e_sarimax_seas),
    ("SARIMAX(1,1,1)", "SARIMAX(1,1,1)(1,0,0)₁₂", e_noseas_seas, e_sarimax_seas),
]

dm_exog_rows = []
for name1, name2, errors1, errors2 in comparisons:
    dm_stat, p_val = diebold_mariano_test(errors1, errors2, h=1)
    if not np.isnan(dm_stat):
        melhor = name2 if dm_stat > 0 else name1
    else:
        melhor = "—"

    dm_exog_rows.append({
        "Modelo 1": name1,
        "Modelo 2": name2,
        "DM stat": round(dm_stat, 4) if not np.isnan(dm_stat) else np.nan,
        "p-valor": round(p_val, 4) if not np.isnan(p_val) else np.nan,
        "Significativo (5%)": "Sim" if not np.isnan(p_val) and p_val < 0.05 else "Não",
        "Melhor": melhor,
    })

dm_exog = pd.DataFrame(dm_exog_rows)
print("Teste de Diebold-Mariano (bilateral, loss=MAE, correção HLN)")
print("DM > 0 => Modelo 2 é melhor | DM < 0 => Modelo 1 é melhor\n")
dm_exog

Teste de Diebold-Mariano (bilateral, loss=MAE, correção HLN)
DM > 0 => Modelo 2 é melhor | DM < 0 => Modelo 1 é melhor



,Modelo 1,Modelo 2,DM stat,p-valor,Significativo (5%),Melhor
0,"ARIMA(1,1,0)","SARIMAX(1,1,1)",0.7738,0.4408,Não,"SARIMAX(1,1,1)"
1,"ARIMA(1,1,0)","SARIMAX(1,1,1)(1,0,0)₁₂",1.2099,0.2291,Não,"SARIMAX(1,1,1)(1,0,0)₁₂"
2,"SARIMAX(1,1,1)","SARIMAX(1,1,1)(1,0,0)₁₂",0.8482,0.3983,Não,"SARIMAX(1,1,1)(1,0,0)₁₂"


Nenhuma das diferenças entre os modelos com exógenas e o ARIMA univariado é estatisticamente significativa a 5%:

- **ARIMA vs SARIMAX com exógenas** (p=0.44): Embora o SARIMAX(1,1,1) tenha MAE menor (0.0749 vs 0.0779), a diferença **não é significativa**. As variáveis macroeconômicas (selic + juros PF) não melhoram o ARIMA de forma estatisticamente comprovada.

- **ARIMA vs SARIMAX sazonal** (p=0.23): Mesmo adicionando componente sazonal, o ganho (MAE 0.0714 vs 0.0779) **não é significativo** a 5%.

- **SARIMAX vs SARIMAX sazonal** (p=0.40): A componente sazonal tampouco melhora significativamente o modelo com exógenas.

Esse resultado é consistente com o que a EDA já sinalizava: o AR(1) com coeficiente ~0.97 captura ~96% da variância da série. As variáveis macro competem pelos ~4% restantes, margem estreita demais para gerar ganho estatisticamente significativo com 105 observações.

Portando os modelos SARIMAX apresentam MAE numericamente menor, mas o teste de Diebold-Mariano não permite rejeitar a hipótese de que essa melhoria é artefato amostral. Esse é, por si só, **um resultado válido e importante**, responde diretamente à pergunta de pesquisa do TCC: variáveis macroeconômicas não demonstram ganho preditivo estatisticamente robusto sobre o ARIMA univariado para a inadimplência PF, dado o forte componente autorregressivo da série.

## Modelos de Machine Learning

Nesta seção avaliamos **Elastic Net** e **XGBoost** usando o mesmo protocolo walk-forward (janela expansível) dos baselines e SARIMAX.

### Estratégia de features: abordagem dual

Para avaliar empiricamente se a curadoria baseada na EDA agrega valor, rodamos cada modelo de ML em **dois cenários**:

1. **Features curadas (EDA-informed)**: as 8 features do `base_modelagem_2`, construídas com lags ótimos do CCF e apenas variáveis do Grupo 1 (estáveis entre regimes). Este cenário testa a hipótese de que parcimônia melhora a generalização, coerente com o achado do SARIMAX (onde 2 exógenas > todas).

2. **Features completas (53)**: a feature matrix de `src/features.py` com lags uniformes {1,3,6} e deltas {1,3} para todas as 10 variáveis exógenas + AR lags. Este cenário delega a seleção de variáveis inteiramente aos algoritmos (regularização L1 no ElasticNet, ganho de informação no XGBoost).

A comparação direta entre os dois cenários responde à pergunta: **a análise exploratória humana agrega valor sobre a seleção automática dos modelos?**

> **Nota sobre dimensionalidade**: Com ~130 observações de treino inicial e 53 features, a razão features/obs é ~0.41. Mesmo com regularização, essa é uma razão alta que favorece a hipótese de que o cenário curado (8 features, razão ~0.06) terá melhor generalização.

In [41]:
# Cenário 1: Features curadas (EDA-informed, base_modelagem_2)
df_curated = pd.read_parquet("../data/processed/base_modelagem_2.parquet")
CURATED_FEATURES = [c for c in df_curated.columns if c != TARGET]

print("=" * 60)
print("CENÁRIO 1 — Features curadas (base_modelagem_2)")
print("=" * 60)
print(f"  Shape: {df_curated.shape}")
print(f"  Features ({len(CURATED_FEATURES)}): {CURATED_FEATURES}")
print(f"  Período: {df_curated.index.min().date()} a {df_curated.index.max().date()}")
print(f"  NaN: {df_curated.isna().sum().sum()}")

# Cenário 2: Features completas (53, pipeline src/features.py)
panel = pd.read_parquet("../data/processed/panel.parquet")
X_full, y_full = build_features(panel)

# Dropar linhas com NaN no target E nas features (evita fillna com mediana)
valid_mask = y_full.notna() & X_full.notna().all(axis=1)
X_full_valid = X_full[valid_mask]
y_full_valid = y_full[valid_mask]

print(f"\n{'=' * 60}")
print("CENÁRIO 2 — Features completas (build_features)")
print("=" * 60)
print(f"  Shape: {X_full_valid.shape}")
print(f"  Período: {y_full_valid.index.min().date()} a {y_full_valid.index.max().date()}")

# Período de teste comum
# Usar o início mais tardio entre os dois cenários para garantir comparabilidade
common_test_start = max(
    df_curated.index[MIN_TRAIN_MONTHS],
    X_full_valid.index[MIN_TRAIN_MONTHS],
)
# Fim comum: menor das duas séries
common_test_end = min(df_curated.index.max(), y_full_valid.index.max())

# Também alinhar com o período dos baselines/SARIMAX (test_dates do df_model)
common_test_start = max(common_test_start, test_dates.min())
common_test_end = min(common_test_end, test_dates.max())

print(f"\n{'=' * 60}")
print("PERÍODO DE TESTE COMUM (todos os modelos)")
print("=" * 60)
print(f"  Início: {common_test_start.date()}")
print(f"  Fim:    {common_test_end.date()}")

# Datas de teste comuns
common_test_dates = df_curated.index[
    (df_curated.index >= common_test_start) & (df_curated.index <= common_test_end)
]
print(f"  N previsões: {len(common_test_dates)}")

2026-03-26 00:34:48 [INFO] src.features: Feature matrix: 314 rows x 53 cols, target: 179 non-null values


CENÁRIO 1 — Features curadas (base_modelagem_2)
  Shape: (153, 9)
  Features (8): ['y_lag1', 'selic_lag8', 'desemprego_lag9', 'ibc_lag12', 'juros_pf_lag6', 'saldo_log_lag1', 'inad_total_lag1', 'cambio_lag1']
  Período: 2013-04-01 a 2025-12-01
  NaN: 0

CENÁRIO 2 — Features completas (build_features)
  Shape: (160, 53)
  Período: 2012-10-01 a 2026-01-01

PERÍODO DE TESTE COMUM (todos os modelos)
  Início: 2017-04-01
  Fim:    2025-12-01
  N previsões: 105


In [42]:
# Helper: walk-forward genérico para modelos de ML
# Evita duplicação de código entre ElasticNet e XGBoost

def walkforward_ml(model_fn, X_data, y_data, test_dates_wf, scale=False):
    """
    Walk-forward expanding window para modelos sklearn-like.

    Args:
        model_fn: callable que retorna um modelo fitted dado (X_train, y_train)
        X_data: DataFrame de features (sem NaN)
        y_data: Series target (sem NaN)
        test_dates_wf: DatetimeIndex das datas de teste
        scale: se True, aplica StandardScaler a cada fold

    Returns:
        results_df, last_model, last_features
    """
    all_dates = y_data.index
    y_true_list, y_pred_list, dates_list = [], [], []
    last_model, last_features = None, None

    for t in test_dates_wf:
        if t not in X_data.index:
            continue
        train_idx = all_dates[all_dates < t]
        if len(train_idx) < MIN_TRAIN_MONTHS:
            continue

        X_train = X_data.loc[train_idx]
        y_train = y_data.loc[train_idx]
        X_test = X_data.loc[[t]]

        if scale:
            scaler = StandardScaler()
            X_train_s = pd.DataFrame(
                scaler.fit_transform(X_train),
                columns=X_train.columns, index=X_train.index,
            )
            X_test_s = pd.DataFrame(
                scaler.transform(X_test),
                columns=X_test.columns, index=X_test.index,
            )
        else:
            X_train_s, X_test_s = X_train, X_test

        model = model_fn(X_train_s, y_train)
        pred = float(model.predict(X_test_s)[0])

        y_true_list.append(float(y_data.loc[t]))
        y_pred_list.append(pred)
        dates_list.append(t)
        last_model = model
        last_features = list(X_train.columns)

    results_df = pd.DataFrame(
        {"y_true": y_true_list, "y_pred": y_pred_list},
        index=pd.to_datetime(dates_list),
    )
    return results_df, last_model, last_features

## Elastic Net

Elastic Net combina regularização L1 (Lasso) e L2 (Ridge), com seleção automática de `alpha` e `l1_ratio` via cross-validation temporal. A regularização L1 promove esparsidade — zerando coeficientes de features irrelevantes — o que é especialmente relevante no cenário com 53 features.

Walk-forward: em cada mês *t*, treina com dados até *t*−1 (com `StandardScaler` re-ajustado a cada passo). Rodamos ambos os cenários (curado e completo) para comparação direta.

In [43]:
# Elastic Net — walk-forward expanding window (dois cenários)

def fit_elasticnet(X_train, y_train):
    """Treina ElasticNet com CV temporal."""
    n_splits = min(5, max(2, len(X_train) // 12))
    model = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99],
        cv=TimeSeriesSplit(n_splits=n_splits),
        max_iter=10000,
    )
    model.fit(X_train, y_train)
    return model

# --- Cenário 1: Features curadas ---
print("ElasticNet — Cenário 1 (features curadas)...")
X_cur = df_curated[CURATED_FEATURES]
y_cur = df_curated[TARGET]
results_enet_curated, last_enet_cur, last_enet_cur_feats = walkforward_ml(
    fit_elasticnet, X_cur, y_cur, common_test_dates, scale=True,
)

# --- Cenário 2: Features completas (53) ---
print("ElasticNet — Cenário 2 (53 features)...")
results_enet_full, last_enet_full, last_enet_full_feats = walkforward_ml(
    fit_elasticnet, X_full_valid, y_full_valid, common_test_dates, scale=True,
)

# Métricas
for name, res in [("Curado (8 feat)", results_enet_curated), ("Completo (53 feat)", results_enet_full)]:
    mae = mean_absolute_error(res["y_true"], res["y_pred"])
    r = rmse(res["y_true"].values, res["y_pred"].values)
    da = directional_accuracy(res["y_true"].values, res["y_pred"].values)
    print(f"\n  ElasticNet {name}: MAE={mae:.4f} | RMSE={r:.4f} | DA={da:.4f} | N={len(res)}")

# Features não-zero do cenário completo (última iteração)
coefs_full = pd.Series(last_enet_full.coef_, index=last_enet_full_feats)
nonzero_full = coefs_full[coefs_full != 0]
print(f"\n  ElasticNet (53 feat): {len(nonzero_full)} features com coeficiente não-zero de {len(coefs_full)}")
print(f"  Features selecionadas:\n{nonzero_full.abs().sort_values(ascending=False).to_string()}")

ElasticNet — Cenário 1 (features curadas)...
ElasticNet — Cenário 2 (53 features)...

  ElasticNet Curado (8 feat): MAE=0.0831 | RMSE=0.1076 | DA=0.5865 | N=105

  ElasticNet Completo (53 feat): MAE=0.0835 | RMSE=0.1091 | DA=0.5673 | N=105

  ElasticNet (53 feat): 6 features com coeficiente não-zero de 53
  Features selecionadas:
inadimplencia_pf_total_ar_lag1              0.467648
selic_acumulada_mes_lag1                    0.024956
ibc_br_dessaz_lag3                          0.013787
pmc_volume_vendas_dessazonalizado_delta1    0.005520
taxa_juros_pf_total_delta3                  0.002429
inadimplencia_carteira_total_delta3         0.002292


## XGBoost

Gradient boosting com árvores de decisão, capaz de capturar relações não-lineares e interações entre variáveis. Hiperparâmetros selecionados via nested `TimeSeriesSplit` CV a cada passo do walk-forward.

Diferenças em relação à versão anterior:
- **Regularização explícita** (`reg_alpha`, `reg_lambda`, `subsample`, `colsample_bytree`) — essencial para a razão features/obs alta do cenário completo
- **Early stopping** para evitar overfitting
- **Dois cenários** (curado vs. completo) para comparação direta

In [29]:
# XGBoost — walk-forward expanding window com nested CV (dois cenários)

# Grid reduzido: 8 combinações (vs. 64 anteriores)
# Foco nos parâmetros de maior impacto: profundidade e regularização
PARAM_GRID_XGB = [
    {"n_estimators": 200, "max_depth": d, "learning_rate": 0.05,
     "reg_alpha": ra, "reg_lambda": rl,
     "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 3}
    for d in [3, 5]
    for ra in [0, 0.1]
    for rl in [1, 5]
]

# Tuning a cada 6 meses (não a cada mês) — reduz de 105 a ~18 rodadas de CV
RETUNE_EVERY = 6

def fit_xgboost_nested_cv(X_train, y_train, cached_params=[None]):
    """Treina XGBoost, re-otimizando hiperparâmetros a cada RETUNE_EVERY chamadas."""
    cached_params[0] = getattr(fit_xgboost_nested_cv, '_call_count', 0)
    fit_xgboost_nested_cv._call_count = cached_params[0] + 1

    # Só faz CV quando necessário
    if cached_params[0] % RETUNE_EVERY == 0 or not hasattr(fit_xgboost_nested_cv, '_best'):
        n_splits = min(3, max(2, len(X_train) // 12))
        tscv = TimeSeriesSplit(n_splits=n_splits)

        best_score, best_params = np.inf, PARAM_GRID_XGB[0]
        for params in PARAM_GRID_XGB:
            scores = []
            for tr_idx, val_idx in tscv.split(X_train):
                xgb = XGBRegressor(**params, random_state=42, verbosity=0,
                                   early_stopping_rounds=20)
                xgb.fit(
                    X_train.iloc[tr_idx], y_train.iloc[tr_idx],
                    eval_set=[(X_train.iloc[val_idx], y_train.iloc[val_idx])],
                    verbose=False,
                )
                preds_cv = xgb.predict(X_train.iloc[val_idx])
                scores.append(mean_absolute_error(y_train.iloc[val_idx], preds_cv))
            avg = np.mean(scores)
            if avg < best_score:
                best_score, best_params = avg, params
        fit_xgboost_nested_cv._best = best_params

    final_model = XGBRegressor(**fit_xgboost_nested_cv._best, random_state=42, verbosity=0)
    final_model.fit(X_train, y_train, verbose=False)
    return final_model

# --- Cenário 1: Features curadas ---
print("XGBoost — Cenário 1 (features curadas)...")
fit_xgboost_nested_cv._call_count = 0  # resetar counter
results_xgb_curated, last_xgb_cur, last_xgb_cur_feats = walkforward_ml(
    fit_xgboost_nested_cv, X_cur, y_cur, common_test_dates, scale=False,
)

# --- Cenário 2: Features completas (53) ---
print("XGBoost — Cenário 2 (53 features)...")
fit_xgboost_nested_cv._call_count = 0  # resetar counter
results_xgb_full, last_xgb_full, last_xgb_full_feats = walkforward_ml(
    fit_xgboost_nested_cv, X_full_valid, y_full_valid, common_test_dates, scale=False,
)

# Métricas
for name, res in [("Curado (8 feat)", results_xgb_curated), ("Completo (53 feat)", results_xgb_full)]:
    mae = mean_absolute_error(res["y_true"], res["y_pred"])
    r = rmse(res["y_true"].values, res["y_pred"].values)
    da = directional_accuracy(res["y_true"].values, res["y_pred"].values)
    print(f"\n  XGBoost {name}: MAE={mae:.4f} | RMSE={r:.4f} | DA={da:.4f} | N={len(res)}")

XGBoost — Cenário 1 (features curadas)...
XGBoost — Cenário 2 (53 features)...

  XGBoost Curado (8 feat): MAE=0.1113 | RMSE=0.1411 | DA=0.5385 | N=105

  XGBoost Completo (53 feat): MAE=0.1067 | RMSE=0.1593 | DA=0.5481 | N=105


### Teste de Diebold-Mariano: ML vs ARIMA(1,1,0)

Seguindo o mesmo protocolo adotado para os modelos SARIMAX, aplicamos o teste de Diebold-Mariano para verificar se os modelos de ML (em ambos os cenários de features) superam o benchmark ARIMA(1,1,0) de forma estatisticamente significativa. Incluímos também a comparação curado vs. completo para cada algoritmo.

In [47]:
# Teste de Diebold-Mariano: ML vs ARIMA(1,1,0)

# Erros do ARIMA no período comum
e_arima_ml = results_arima110.loc[
    results_arima110.index.isin(common_test_dates)
]
e_arima_arr = e_arima_ml["y_true"].values - e_arima_ml["y_pred"].values

ml_models = {
    "ElasticNet (curado)": results_enet_curated,
    "ElasticNet (53 feat)": results_enet_full,
    "XGBoost (curado)": results_xgb_curated,
    "XGBoost (53 feat)": results_xgb_full,
}

dm_ml_rows = []

# Parte 1: cada modelo ML vs ARIMA
for name, res in ml_models.items():
    e_model = res["y_true"].values - res["y_pred"].values
    min_len = min(len(e_arima_arr), len(e_model))
    dm_stat, p_val = diebold_mariano_test(e_arima_arr[:min_len], e_model[:min_len], h=1)
    melhor = name if dm_stat > 0 else "ARIMA(1,1,0)"
    dm_ml_rows.append({
        "Modelo 1": "ARIMA(1,1,0)",
        "Modelo 2": name,
        "DM stat": round(dm_stat, 4),
        "p-valor": round(p_val, 4),
        "Significativo (5%)": "Sim" if p_val < 0.05 else "Não",
        "Melhor": melhor,
    })

# Parte 2: curado vs completo
for algo in ["ElasticNet", "XGBoost"]:
    e_cur = ml_models[f"{algo} (curado)"]["y_true"].values - ml_models[f"{algo} (curado)"]["y_pred"].values
    e_full = ml_models[f"{algo} (53 feat)"]["y_true"].values - ml_models[f"{algo} (53 feat)"]["y_pred"].values
    min_len = min(len(e_cur), len(e_full))
    dm_stat, p_val = diebold_mariano_test(e_full[:min_len], e_cur[:min_len], h=1)
    melhor = f"{algo} (curado)" if dm_stat > 0 else f"{algo} (53 feat)"
    dm_ml_rows.append({
        "Modelo 1": f"{algo} (53 feat)",
        "Modelo 2": f"{algo} (curado)",
        "DM stat": round(dm_stat, 4),
        "p-valor": round(p_val, 4),
        "Significativo (5%)": "Sim" if p_val < 0.05 else "Não",
        "Melhor": melhor,
    })

dm_ml = pd.DataFrame(dm_ml_rows)
display(dm_ml)

,Modelo 1,Modelo 2,DM stat,p-valor,Significativo (5%),Melhor
0,"ARIMA(1,1,0)",ElasticNet (curado),-1.3967,0.1655,Não,"ARIMA(1,1,0)"
1,"ARIMA(1,1,0)",ElasticNet (53 feat),-1.2641,0.2090,Não,"ARIMA(1,1,0)"
2,"ARIMA(1,1,0)",XGBoost (curado),-5.3415,0.0000,Sim,"ARIMA(1,1,0)"
3,"ARIMA(1,1,0)",XGBoost (53 feat),-3.2041,0.0018,Sim,"ARIMA(1,1,0)"
4,ElasticNet (53 feat),ElasticNet (curado),0.0834,0.9337,Não,ElasticNet (curado)
5,XGBoost (53 feat),XGBoost (curado),-0.5029,0.6161,Não,XGBoost (53 feat)


#### 1. Nenhum modelo de ML supera o ARIMA(1,1,0) univariado

| Comparação | DM stat | p-valor | Resultado |
|---|:---:|:---:|---|
| ARIMA vs ElasticNet (curado) | −1.40 | 0.166 | Diferença **não significativa** |
| ARIMA vs ElasticNet (53 feat) | −1.26 | 0.209 | Diferença **não significativa** |
| ARIMA vs XGBoost (curado) | −5.34 | <0.001 | ARIMA **significativamente melhor** |
| ARIMA vs XGBoost (53 feat) | −3.20 | 0.002 | ARIMA **significativamente melhor** |

O sinal negativo do DM stat indica que o ARIMA tem erros menores em todos os casos. Para o ElasticNet, a diferença não é estatisticamente significativa, ou seja, o ElasticNet é "quase tão bom" quanto o ARIMA, mas não o supera. Para o XGBoost, a diferença é forte e significativa: o XGBoost é **significativamente pior** que o ARIMA univariado.

**Por que o XGBoost perdeu?** Árvores de decisão precisam de dados suficientes para aprender partições úteis. Com ~48–130 observações de treino (janela expansível), o XGBoost provavelmente memorizou ruído em vez de aprender padrões genuínos. Em séries temporais curtas com forte componente AR(1), modelos não-paramétricos tendem a perder para modelos lineares parcimoniosos.

#### 2. Curado vs. Completo, a EDA faz diferença?

| Comparação | DM stat | p-valor | Resultado |
|---|:---:|:---:|---|
| ElasticNet (53) vs ElasticNet (curado) | 0.08 | 0.934 | **Sem diferença** |
| XGBoost (53) vs XGBoost (curado) | −0.50 | 0.616 | **Sem diferença** |

A curadoria de features baseada na EDA **não melhora nem piora** os modelos de ML de forma significativa. Isso admite duas leituras complementares:

- **Para o ElasticNet**: a regularização L1 já "redescobriu" as mesmas variáveis que a EDA indicou (apenas 6 de 53 features não-zero, todas do Grupo 1 estável). Ou seja, a EDA e o L1 chegaram ao **mesmo resultado por caminhos independentes** a curadoria manual é redundante quando há regularização adequada, mas valida a análise exploratória.

- **Para o XGBoost**: mesmo com features curadas, o modelo não melhora significativamente. O problema não são as features, mas sim a **inadequação do algoritmo** para esta tarefa específica (série curta, dinâmica dominada por AR(1)).

---

#### 3. Hierarquia consolidada com todos os testes DM

Combinando os três blocos de testes DM realizados ao longo do notebook:

```
ARIMA(1,1,0) ─── benchmark ──────────────────────────
    │
    ├─ > Persistência      (DM=2.60, p=0.011) ✓ sig
    ├─ > AR(1)             (DM=2.86, p=0.005) ✓ sig
    │
    ├─ ≈ SARIMAX(1,1,1)    (DM=0.77, p=0.441) ✗ n.s.
    ├─ ≈ SARIMAX sazonal   (DM=1.21, p=0.229) ✗ n.s.
    │
    ├─ ≈ ElasticNet        (DM=−1.40, p=0.166) ✗ n.s.
    │
    ├─ > XGBoost (curado)  (DM=−5.34, p<0.001) ✓ sig
    └─ > XGBoost (53 feat) (DM=−3.20, p=0.002) ✓ sig
```

Legenda: `>` = ARIMA é significativamente melhor; `≈` = diferença não significativa.

#### 4. Implicações para a questão de pesquisa do TCC

A questão central é: *"variáveis macroeconômicas melhoram a previsão da inadimplência PF?"*

A resposta empírica é **nuançada**:

1. **Não melhoram de forma estatisticamente significativa.** Nenhum modelo com exógenas (SARIMAX, ElasticNet, XGBoost) superou o ARIMA univariado no teste DM a 5%.

2. **Também não pioram significativamente** (exceto XGBoost). O SARIMAX e o ElasticNet com exógenas ficam na zona de indiferença estatística, as variáveis macro não adicionam ruído suficiente para degradar o modelo, mas tampouco adicionam sinal suficiente para melhorá-lo.

3. **A dominância do AR(1)** (~96% da variância) é o fator explicativo central. A inadimplência PF é uma série altamente persistente, o valor do mês passado é o melhor preditor do valor deste mês, e as variáveis macroeconômicas competem por uma margem residual de ~4%.

4. **Esse resultado é válido e publicável.** Demonstrar que variáveis exógenas não agregam valor preditivo é tão informativo quanto demonstrar que agregam, responde diretamente à questão de pesquisa e tem implicações práticas para gestores de risco (um modelo univariado simples é suficiente para nowcasting de curto prazo).